In [ ]:
from ugot import ugot
got = ugot.UGOT()
got.initialize("10.210.71.170")
got.load_models(["apriltag_qrcode"])
got.open_camera()

got.balance_start_balancing()
tags = got.get_apriltag_total_info()
input() # wait for user input before moving

# move forward slowly while apriltag not seen
while not tags:
    got.balance_move_speed(0, 12.5)
    tags = got.get_apriltag_total_info()

# approach at full speed when apriltag seen
while True:
    got.balance_move_speed(0, 80)
    tags = got.get_apriltag_total_info()
    if not tags:
        break

# decelerate
for i in range(80, 0, -5):
    got.balance_move_speed(0, i)

got.balance_stop_balancing()

def AP_centralization_approaching(distance=0.15, gap=20, fwd_spd=10, strafe_spd=10):
    """
    Drive toward a detected AprilTag, keeping it centered in the camera frame.

    Parameters:
        distance  (float): Stop when the tag is within this many meters (default 0.15 m).
        gap       (int):   Pixel tolerance around center (320 px) before strafing (default 20 px).
        fwd_spd   (int):   Forward drive speed percentage (default 10 cm/s).
        strafe_spd(int):   Left/right correction speed percentage (default 10 cm/s).
    """
    # Get an initial reading to confirm a tag is visible before entering the loop.
    AP_info = got.get_apriltag_total_info()
    AP_x = AP_info[0][1] # Horizontal pixel position of the tag (0=left, 640=right)
    AP_distance = AP_info[0][6] # Estimated distance to the tag in meters

    while True:
        # Refresh tag data every iteration for responsive corrections.
        AP_info = got.get_apriltag_total_info()
        AP_x = AP_info[0][1]
        AP_distance = AP_info[0][6]

        if AP_x < 320 - gap:
            # Tag is to the LEFT of center — strafe left to re-align.
            # mecanum_move_xyz(x, y, z): x=strafe, y=forward, z=rotation
            got.mecanum_move_xyz(-strafe_spd, strafe_spd, 0)
        elif AP_x > 320 + gap:
            # Tag is to the RIGHT of center — strafe right to re-align.
            got.mecanum_move_xyz(strafe_spd, strafe_spd, 0)
        elif AP_distance > distance:
            # Tag is centered but still too far — drive straight forward.
            got.mecanum_move_xyz(0, fwd_spd, 0)
        else:
            # Tag is centered AND within target distance — stop and exit.
            got.mecanum_stop()
            print("It's too close, let's stop.")
            break

try:
    while True:
        # Poll the camera for any visible AprilTags.
        tag = got.get_apriltag_total_info()

        if tag:
            # A tag is visible — approach it and pick up the object.
            # Lower speeds (fwd_spd=5, strafe_spd=7) for more precise final approach.
            AP_centralization_approaching(distance=0.15, gap=20, fwd_spd=5, strafe_spd=7)

            print("Approached!")
            break  # Exit after one successful pick-and-place

except KeyboardInterrupt:
    # Safety stop if the cell is interrupted manually.
    got.mecanum_stop()
    print("Done")


10.210.71.170:50051
